<a href="https://colab.research.google.com/github/diego-ascenciorod/Clase-Laboratorio-Estadistico/blob/main/A06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report

# cargar datos
df = pd.read_csv("Heart Prediction Quantum Dataset.csv")

# ver datos
print(df.head())
print(df.info())
print(df.isnull().sum())

# ver clases
print(df["HeartDisease"].value_counts())


# EDA

cols = ["Age","BloodPressure","Cholesterol","HeartRate","QuantumPatternFeature"]

for col in cols:
    plt.hist(df[col], bins=15)
    plt.title(col)
    plt.show()

for col in cols:
    df.boxplot(column=col, by="HeartDisease")
    plt.title(col)
    plt.suptitle("")
    plt.show()

print(df.corr())


# X y Y

X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# preprocessing
# no use labelencoder porque ya todo es numerico

scaler = StandardScaler()
X = scaler.fit_transform(X)

# kfold + buscar C

kf = KFold(n_splits=5, shuffle=True, random_state=42)

valores_C = [0.01, 0.1, 1, 10]

mejor_C = 0
mejor_f1 = 0

for c in valores_C:
    modelo = SVC(C=c, kernel="linear")

    scores = cross_val_score(modelo, X, y, cv=kf, scoring="f1")

    prom = scores.mean()
    print("C:", c, "F1:", prom)

    if prom > mejor_f1:
        mejor_f1 = prom
        mejor_C = c

print("mejor C:", mejor_C)

# train test

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# modelo final
modelo = SVC(C=mejor_C, kernel="linear")
modelo.fit(X_train, y_train)

# predicciones
y_pred = modelo.predict(X_test)

# matriz de confusion

cm = confusion_matrix(y_test, y_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()

# metricas a mano (como en el pizarron)
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
specificity = tn / (tn + fp)
f1 = (2 * precision * recall) / (precision + recall)

print("accuracy:", accuracy)
print("precision:", precision)
print("recall:", recall)
print("specificity:", specificity)
print("f1:", f1)

# con sklearn
print("accuracy sklearn:", accuracy_score(y_test, y_pred))
print("f1 sklearn:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))